# Knowledge Graph Construction

In [ ]:
# Import required packages
import json
import os
import dotenv
import requests
import pandas as pd
import yatter
import subprocess
from ruamel.yaml import YAML
from src import SPARQL
import isodate

dotenv.load_dotenv("secrets.env") # Load environment variables from a file, a Manifesto Project secret is required as MANIFESTO_API_KEY


## 1. Data Acquisition

In [ ]:
mappings_data_path= "mappings"
manifestos_data_path= "data/kg_data"
manifesto_project_api_base_url = "https://manifesto-project.wzb.eu/api/v1/"

# Elections dates in wikipedia and the date in manifesto projects
elections= {
            "FR": {
                    "2022-06": "2022-06-12",
                    "2017-06": "2017-06-11",
                    "2012-06": "2012-06-10",
                },

            "ES": {
                    "2023-07": "2023-07-23",
                    "2019-11": "2019-11-10",
                    "2019-04": "2019-04-28",
                    "2016-06": "2016-06-26",
                    "2015-12": "2015-12-20",
                    "2011-11": "2011-11-20"
                }
}

# Area Served
isoarea2wd= {
            "ES": "http://www.wikidata.org/entity/Q29",
            "FR": "http://www.wikidata.org/entity/Q142"
        }

wd2isoarea= {
            "http://www.wikidata.org/entity/Q29": "ES",
            "http://www.wikidata.org/entity/Q142": "FR"
        }

In [ ]:
# Link the areas and the agents
areas= []

for area, wd_id_area in isoarea2wd.items():
    areas.append({
        "area_name": area,
        "area_wd_id": wd_id_area
    })

areas= pd.DataFrame(areas)
os.makedirs(f"{mappings_data_path}/resources", exist_ok=True)

areas.to_csv(f"{mappings_data_path}/resources/areas.csv", index=False)

In [ ]:
# Downloading manifestos data

with open("wd2mp.json", "r") as f:
    wd2mp = json.load(f)
    

def download_manifesto(url: str, params:dict, output_path: str):
    """Download a Manifesto Project resource using your API key."""
    headers={"Authorization": f"Bearer {os.getenv('MANIFESTO_API_KEY')}"}
    r = requests.get(url, headers=headers, params=params, timeout=60)
    if r.status_code != 200:
        raise RuntimeError(f"Error {r.status_code} when downloading {url}")
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    with open(output_path, "wb") as f:
        f.write(r.content)
    
    print(f"✅ Saved to {output_path}")


for wd_id_agent, agent in wd2mp.items():
    
    os.makedirs(f"{manifestos_data_path}/{agent['country']}", exist_ok=True)
    os.makedirs(f"{manifestos_data_path}/{agent['country']}/{wd_id_agent}", exist_ok=True)

    for election_id, election_date in elections[agent["country"]].items():
        download_url= f"{manifesto_project_api_base_url}/texts_and_annotations"
        params = {
                "version": "2025-1",
                "keys[]": f"{agent['Manifesto Project ID']}_{election_id.replace('-', '')}"  # requests will repeat this param for each key
            }
        
        output_path= f"{manifestos_data_path}/{agent['country']}/{wd_id_agent}/{election_date}.json"
        
        if not os.path.exists(output_path):
            download_manifesto(download_url, params, output_path)
            continue
        
        print(f"+ {output_path} already exists, skipping download.")
        
        

In [ ]:
# Transforming manifestos data to useful CSV
with open("wd2mp.json", "r") as f:
    wd2mp = json.load(f)
    
manifestos= []
proposals= []
for root, _, files in os.walk(manifestos_data_path):
    for file in files:
        if file.endswith(".json"):
            parts = root.split(os.sep)
            
            if len(parts) < 2:  # Ensure there are at least two levels
                raise ValueError(f"Unexpected directory structure: {root}")
                
            country = parts[-2]  # Parent folder (e.g., "fra", "spa")
            agent = parts[-1]  # Current folder (e.g., "Q170972")
                
            with open(f"{root}/{file}", "r") as f:
                data = json.load(f)
            
            if data["items"]:
                manifestos.append({
                                    "area": country, 
                                    "AgentUri": f"http://www.wikidata.org/entity/{agent}",
                                    "AgentName": wd2mp[agent]['label'],
                                    "Description": "Party manifesto of " + wd2mp[agent]['label'],
                                    "Created": file.replace(".json", ""), 
                                    "Source": "Manifesto Project", 
                                    "PartiManifestoId": f"{agent}-{file.replace('.json', '')}"
                                })
                
                for item_num, item in enumerate(data["items"][0]["items"]):
                    if "text" in item:
                
                        proposals.append({
                                    "text": item["text"],
                                    "length": len(item["text"].split()),
                                    "Source": "Manifesto Project", 
                                    "Created": file.replace(".json", ""), 
                                    "area": country, 
                                    "PartiManifestoId": f"{agent}-{file.replace('.json', '')}",
                                    "AgentUri": f"http://www.wikidata.org/entity/{agent}", 
                                    "Description": "Proposal of the party manifesto of " + wd2mp[agent]['label'],
                                    "ProposalId": f"{agent}-{file.replace('.json', '')}-{item_num}"
                                })
                
        
manifestos = pd.DataFrame(manifestos)
manifestos["Created"] = pd.to_datetime(manifestos["Created"])  # Convert to datetime
manifestos["Created"] = manifestos["Created"].dt.tz_localize("UTC")  # Assume UTC
manifestos["Created"] = manifestos["Created"].dt.strftime("%Y-%m-%dT%H:%M:%SZ")

proposals = pd.DataFrame(proposals)
proposals["Created"] = pd.to_datetime(proposals["Created"])  # Convert to datetime
proposals["Created"] = proposals["Created"].dt.tz_localize("UTC")  # Assume UTC
proposals["Created"] = proposals["Created"].dt.strftime("%Y-%m-%dT%H:%M:%SZ")

manifestos.to_csv(f"{mappings_data_path}/resources/manifestos.csv", index=False)
proposals.to_csv(f"{mappings_data_path}/resources/proposals.csv", index=False)

## 2. Knowledge Fusion

In [ ]:
# Elections dates retrieved from wikidata and the creation of the elections and campaign intervals
# Discourses are also mapped to these events

# Prefixes dict of all the KG
prefixes= { "dc": "http://purl.org/dc/elements/1.1/",
            "dcam": "http://purl.org/dc/dcam/",
            "eli": "http://data.europa.eu/eli/ontology#",
            "foaf": "http://xmlns.com/foaf/0.1/",
            "lkg": "http://lkg.lynx-project.eu/def/",
            "nif-core": "http://persistence.uni-leipzig.org/nlp2rdf/ontologies/nif-core#",
            "owl": "http://www.w3.org/2002/07/owl#",
            "podio": "http://w3id.org/podio#",
            "rdf": "http://www.w3.org/1999/02/22-rdf-syntax-ns#",
            "schema": "http://schema.org/",
            "sioc": "http://rdfs.org/sioc/ns#",
            "skos": "http://www.w3.org/2004/02/skos/core#",
            "terms": "http://purl.org/dc/terms/",
            "vann": "http://purl.org/vocab/vann/",
            "xml": "http://www.w3.org/XML/1998/namespace",
            "xsd": "http://www.w3.org/2001/XMLSchema#",
            "rdfs": "http://www.w3.org/2000/01/rdf-schema#",
            "nif": "http://persistence.uni-leipzig.org/nlp2rdf/ontologies/nif-core#",
            "wd": "http://www.wikidata.org/entity/",
            "wdt": "http://www.wikidata.org/prop/direct/",
            "wikibase": "http://wikiba.se/ontology#",
            "bd": "http://www.bigdata.com/rdf#",
            "ps": "http://www.wikidata.org/prop/statement/",
            "p": "http://www.wikidata.org/prop/",
            "pq": "http://www.wikidata.org/prop/qualifier/",
            "ptr": "http://w3id.org/podio/resource/",
            "time": "http://www.w3.org/2006/time#"
            }

# Prefixes declaration during all the sparql queries
PREFIXES = """""".join(f"PREFIX {k}: <{v}>\n" for k, v in prefixes.items()) + """"""

# Wikidata endpoint declaration
WD_ENDPOINT= SPARQL(endpoint="wikidata")

manifestos= pd.read_csv(f"{mappings_data_path}/resources/manifestos.csv")
proposals= pd.read_csv(f"{mappings_data_path}/resources/proposals.csv")

In [ ]:
# Elections

def get_france_general_elections(both_rounds=False):
    """
    Args:
    ----
        both_rounds(bool): Default False. Returns both rounds of elections if True. Else it returns the last round
    """
    if both_rounds:
        query="""SELECT DISTINCT ?electionsLabel (STR(?_date) AS ?date)
            WHERE
            {
            ?elections wdt:P31 wd:Q3587148.
            ?elections p:P585 ?pointInTime .
            ?pointInTime ps:P585 ?_date .
            SERVICE wikibase:label { bd:serviceParam wikibase:language "[AUTO_LANGUAGE],en". }
            # Ensure ?elections is not part of anything (P361)
            FILTER NOT EXISTS { ?elections wdt:P361 ?partOf. }
            }
            GROUP BY ?elections ?electionsLabel ?_date
            ORDER BY DESC(?date)"""
    else:
        query="""SELECT DISTINCT ?electionsLabel (STR(MAX(?_date)) AS ?date)
            WHERE
            {
            ?elections wdt:P31 wd:Q3587148.
            ?elections p:P585 ?pointInTime .
            ?pointInTime ps:P585 ?_date .
            SERVICE wikibase:label { bd:serviceParam wikibase:language "[AUTO_LANGUAGE],en". }
            # Ensure ?elections is not part of anything (P361)
            FILTER NOT EXISTS { ?elections wdt:P361 ?partOf. }
            }
            GROUP BY ?elections ?electionsLabel
            ORDER BY DESC(?date)"""
    
    return WD_ENDPOINT.run_query_pandas(f"{PREFIXES} {query}")

def get_spain_general_elections():
    query= """SELECT ?electionsLabel (STR(?_date) AS ?date)
        WHERE
        {
        ?elections wdt:P31 wd:Q17317594.
        ?elections p:P585 ?pointInTime .
        ?pointInTime ps:P585 ?_date .
        SERVICE wikibase:label { bd:serviceParam wikibase:language "[AUTO_LANGUAGE],en". }
        }
        GROUP BY ?elections ?electionsLabel ?_date
        ORDER BY DESC(?date)"""

    return WD_ENDPOINT.run_query_pandas(f"{PREFIXES} {query}")

def assign_area(label):
    """Retrieve the area code depending on the election label

    Args:
        label (str): Election label

    Returns:
        str: The area code.
    """
    if   "Spanish" in label: elections_id_area = "ES"
    elif "French"  in label: elections_id_area = "FR"
    else: elections_id_area = None
    return elections_id_area


campaign_duration= 15 # Number of days of electoral campaigns, same as previous study

es_elections= get_spain_general_elections()
es_elections["area"]= es_elections["electionsLabel"].apply(assign_area)
fr_ellections= get_france_general_elections(both_rounds=True)
fr_ellections["area"]= fr_ellections["electionsLabel"].apply(assign_area)

all_elections= pd.concat([es_elections, fr_ellections]) # Retrieve elections info from wikidata

# Transform the wikidata response
all_elections["date"]= pd.to_datetime(all_elections["date"])
all_elections= all_elections[all_elections["date"]>="2011-01-01"]  # Filter elections to after 2011
all_elections= all_elections[all_elections["date"]<="2024-01-01"]  # Filter elections to after 2011
all_elections= all_elections.sort_values(by='date', ascending=True).reset_index(drop=True)


In [ ]:
instants= []
elections= [] # datetime_intervals
campaigns= [] # proper_intervals


all_elections= all_elections.sort_values(by=['area','date'], ascending=[True, True]).reset_index(drop=True)

current_area= None
for label, group in all_elections.groupby("electionsLabel", sort=False):
    group= group.reset_index(drop=True)
    if current_area != group['area'].iloc[0]:
        previous_elections= None
        previous_campaign= None
        current_area= group['area'].iloc[0]

    elections_id_area= assign_area(label=label) # Retrieve area code

    election_id= f"Election-{elections_id_area}-{group['date'].max().strftime('%Y-%m')}"
    
    campaign_id= f"Campaign-{elections_id_area}-{group['date'].max().strftime('%Y-%m')}"
    campaign_init= group['date'].min()-pd.Timedelta(days=campaign_duration)#.isoformat()
    campaign_end=  group['date'].max()#.isoformat()

    for row_num, row in group.iterrows():
        if   "Spanish" in label: 
            _elections_id = election_id
            _label= label
        elif "French"  in label: 
            _elections_id = election_id+"r"+str(row_num+1)
            if row_num == 0: 
                _label= "First round of the "+ label
            if row_num == 1: 
                _label= "Second round of the "+ label
        
        # Election happening Instant
        instants.append({
                            "instant": row['date'].strftime('%Y-%m-%d'),# Defining the instant of the election init
                            "datetime": row['date'].isoformat() # Defining the moment of the init
                        })
        # Election Interval        
        _elections_= {
                        "election_id": _elections_id, # Election identifier
                        "begin": row['date'].strftime('%Y-%m-%d'), # Referencing the begining of the election
                        "end": row['date'].strftime('%Y-%m-%d'), # Referencing the end of the election
                        "description": _label, # Elections description
                        "area": elections_id_area, # Referencing the area served for the election
                        "datetime": row['date'].isoformat(), # Elections date
                        "finishes": None, # Referencing the campaign that finishes this election
                        "after": previous_elections, # Referencing the previous election end
                        "type": "Election" # Type of interval Election
                    }

        if row_num == len(group)-1: # Only for the last round of the elections
            _elections_["finishes"]= campaign_id # Referencing the campaign that finishes this election

        if previous_elections is not None:
            _elections_["after"]= previous_elections # Referencing the previous election end
            

        elections.append(_elections_)
        previous_elections= _elections_id # For the next loop

    ###
    ### ---
    ###

    # Campaign init Instant
    instants.append({
                    "instant": campaign_init.strftime('%Y-%m-%d'),# Defining the instant of the campaign init
                    "datetime": campaign_init.isoformat() # Defining the moment of the init
                })
    
    # Campaign end Instant
    instants.append({
                    "instant": campaign_end.strftime('%Y-%m-%d'),# Defining the instant of the campaign init
                    "datetime": campaign_end.isoformat() # Defining the moment of the init
                })
    

    # Campaign Interval
    
    _campaigns_= {
        "campaign_id": campaign_id, # Campaign identifier
        "after": previous_campaign, # Referencing the previous campaign end
        "begin": campaign_init.strftime('%Y-%m-%d'), # Referencing the begining of the campaign
        "end": campaign_end.strftime('%Y-%m-%d'), # Referencing the end of the campaign
        "finished_by": _elections_id, # Referencing the election that finishes this campaign
        "area": elections_id_area, # Referencing the area served for the campaign
        "duration": isodate.duration_isoformat(campaign_end - campaign_init), # Defining the campaign duration
        "description": "Electoral Campaign of the "+ label, # Label of the campaign period
        "type": "Campaign" # Type of interval Campaign
    }
    previous_campaign= campaign_id # For the next loop
    
    campaigns.append(_campaigns_)


elections= pd.DataFrame(elections)
campaigns= pd.DataFrame(campaigns)
instants= pd.DataFrame(instants)

elections.to_csv(f"{mappings_data_path}/resources/elections.csv", index=False)
campaigns.to_csv(f"{mappings_data_path}/resources/campaigns.csv", index=False)
instants.to_csv(f"{mappings_data_path}/resources/instants.csv", index=False)



In [ ]:
proposals= pd.read_csv(f"{mappings_data_path}/resources/proposals.csv")

elections= pd.read_csv(f"{mappings_data_path}/resources/elections.csv")
campaigns= pd.read_csv(f"{mappings_data_path}/resources/campaigns.csv")


discourses= proposals[["ProposalId", "Created", "area"]].rename(columns={"ProposalId": "id"})

intervals= pd.concat(
    [elections[["election_id", "begin", "end", "area"]].rename(columns={"election_id": "id"}),
     campaigns[["campaign_id", "begin", "end", "area"]].rename(columns={"campaign_id": "id"})
     ]
)

discourses["Created"] = pd.to_datetime(discourses["Created"])

intervals["begin"]   = pd.to_datetime(intervals["begin"])
intervals["begin"] = intervals["begin"].dt.tz_localize("UTC")  # Assume UTC
intervals["begin"] = intervals["begin"].dt.strftime("%Y-%m-%dT%H:%M:%SZ")

intervals["end"]     = pd.to_datetime(intervals["end"])
intervals["end"] = intervals["end"].dt.tz_localize("UTC")  # Assume UTC
intervals["end"] = intervals["end"].dt.strftime("%Y-%m-%dT%H:%M:%SZ")



insides = (
    discourses.assign(_k=1)
    .merge(intervals.assign(_k=1), on="_k")
    .loc[lambda df: (df["begin"] <= df["Created"]) & (df["Created"] <= df["end"]),
         ["id_x", "id_y"]]
    .rename(columns={"id_x": "id_discourse", "id_y": "id_interval"})
    .reset_index(drop=True)
)

insides.to_csv(f"{mappings_data_path}/resources/insides.csv", index=False)

## Knowledge Storage

In [ ]:
# Generation of RDF triples from YARRRML mappings
filenames= [
            "manifestos", 
            "proposals",
            "areas",
            "time"
            ]

# YARRRML to RML
yaml = YAML(typ='safe', pure=True)

rml_mapper= f"{mappings_data_path}/rmlmapper.jar"

for filename in filenames:
    rml_content = yatter.translate(yaml.load(open(f"{mappings_data_path}/{filename}.yml")))

    with open(f"{mappings_data_path}/{filename}_rules.rml", "w") as f:
        f.write(rml_content)

os.makedirs(f"{mappings_data_path}/triples", exist_ok=True)

# RML to Triples
for filename in filenames:
    print(f"Generating triples for {filename} file")

    command= [  "java",
                "-jar",
                f"{rml_mapper}",
                "-m",
                f"{mappings_data_path}/{filename}_rules.rml",
                "-o",
                f"{mappings_data_path}/triples/{filename}_triples.nt"
            ]
    subprocess.run(command)

## Stats

For generating the stats is mandatory to build the Knowledge Graph and populate it with the triples and the ontologies.

In [ ]:
# Prefixes dict of all the KG
prefixes= { "dc": "http://purl.org/dc/elements/1.1/",
            "dcam": "http://purl.org/dc/dcam/",
            "eli": "http://data.europa.eu/eli/ontology#",
            "foaf": "http://xmlns.com/foaf/0.1/",
            "lkg": "http://lkg.lynx-project.eu/def/",
            "nif-core": "http://persistence.uni-leipzig.org/nlp2rdf/ontologies/nif-core#",
            "owl": "http://www.w3.org/2002/07/owl#",
            "podio": "http://w3id.org/podio#",
            "rdf": "http://www.w3.org/1999/02/22-rdf-syntax-ns#",
            "schema": "http://schema.org/",
            "sioc": "http://rdfs.org/sioc/ns#",
            "skos": "http://www.w3.org/2004/02/skos/core#",
            "terms": "http://purl.org/dc/terms/",
            "vann": "http://purl.org/vocab/vann/",
            "xml": "http://www.w3.org/XML/1998/namespace",
            "xsd": "http://www.w3.org/2001/XMLSchema#",
            "rdfs": "http://www.w3.org/2000/01/rdf-schema#",
            "nif": "http://persistence.uni-leipzig.org/nlp2rdf/ontologies/nif-core#",
            "wd": "http://www.wikidata.org/entity/",
            "wdt": "http://www.wikidata.org/prop/direct/",
            "wikibase": "http://wikiba.se/ontology#",
            "bd": "http://www.bigdata.com/rdf#",
            "ps": "http://www.wikidata.org/prop/statement/",
            "p": "http://www.wikidata.org/prop/",
            "pq": "http://www.wikidata.org/prop/qualifier/",
            "ptr": "http://w3id.org/podio/resource/",
            "time": "http://www.w3.org/2006/time#"
            }

# Prefixes declaration during all the sparql queries
PREFIXES = """""".join(f"PREFIX {k}: <{v}>\n" for k, v in prefixes.items()) + """"""

# TR-KG endpoint declaration
GDB_REPO= "politicaltr" # Name of the GraphDB repository
GDB_PORT= 7300 # Port of the GraphDB instance
GDB_URI= f"http://0.0.0.0:{GDB_PORT}/repositories/{GDB_REPO}" # Full URI of the GraphDB repository 
GDB_ENDPOINT= SPARQL(endpoint=GDB_URI)

# Query to retrieve manifestos info along with the political alignment of their agents
def get_manifesto_info():
    query= """
    SELECT DISTINCT ?manifesto  ?areaServed ?agentName ?alignment
    WHERE {
        ?manifesto  a podio:PartyManifesto ;
                    schema:areaServed ?areaServed ;
                    terms:publisher ?agent .
        ?agent rdfs:label ?agentName .
        
        SERVICE <https://query.wikidata.org/sparql> {
            ?agent wdt:P1387 ?politicalAlignment .
            ?politicalAlignment rdfs:label ?politicalAlignmentLabel .
            FILTER(LANG(?politicalAlignmentLabel)="en") .
        }
        
        BIND(LCASE(STR(?politicalAlignmentLabel)) AS ?normalizedLabel) .
        
        # Agrupación de categorías políticas
        BIND(
            IF(
                ?normalizedLabel IN ("centre-left", "centre-right", "centrism"), "centrism",
                IF(
                    ?normalizedLabel IN ("far-left", "radical left", "left-wing"), "left-wing",
                    IF(
                        ?normalizedLabel IN ("right-wing", "right-wing extremism", "radical right in europe"), "right-wing",
                        "other"
                    )
                )
            ) AS ?alignment
        )
    } ORDER BY DESC(?timestamp)

    """

    return GDB_ENDPOINT.run_query_pandas(f"{PREFIXES} {query}")

# Query to retrieve proposals info along with the political alignment of their agents
def get_proposals_info():
    query= """
        SELECT DISTINCT ?timestamp ?areaServed ?agentName ?alignment ?content 
        WHERE {
            ?discourse a podio:Discourse .
            ?discourse podio:content ?content .
            ?discourse terms:created ?timestamp .
            ?discourse terms:publisher ?agent .
            ?agent rdfs:label ?agentName .
            ?discourse schema:areaServed ?areaServed .
            
            SERVICE <https://query.wikidata.org/sparql> {
                ?agent wdt:P1387 ?politicalAlignment .
                ?politicalAlignment rdfs:label ?politicalAlignmentLabel .
                FILTER(LANG(?politicalAlignmentLabel)="en") .
            }
            
            BIND(LCASE(STR(?politicalAlignmentLabel)) AS ?normalizedLabel) .
            
            # Agrupación de categorías políticas
            BIND(
                IF(
                    ?normalizedLabel IN ("centre-left", "centre-right", "centrism"), "centrism",
                    IF(
                        ?normalizedLabel IN ("far-left", "radical left", "left-wing"), "left-wing",
                        IF(
                            ?normalizedLabel IN ("right-wing", "right-wing extremism", "radical right in europe"), "right-wing",
                            "other"
                        )
                    )
                ) AS ?alignment
            )
        }
        ORDER BY DESC(?timestamp)
        """

    return GDB_ENDPOINT.run_query_pandas(f"{PREFIXES} {query}")

In [ ]:
manifestos= get_manifesto_info()
print(f"Total manifestos {len(manifestos)}")
print("--"*10)
print(f"Total manifestos for France {len(manifestos[(manifestos['areaServed']=='http://w3id.org/podio/resource/FR')])}")
print(f"Left-wing manifestos for France {len(manifestos[(manifestos['areaServed']=='http://w3id.org/podio/resource/FR') & (manifestos['alignment']=='left-wing')])}")
print(f"Centrism manifestos for France {len(manifestos[(manifestos['areaServed']=='http://w3id.org/podio/resource/FR') & (manifestos['alignment']=='centrism')])}")
print(f"Right-wing manifestos for France {len(manifestos[(manifestos['areaServed']=='http://w3id.org/podio/resource/FR') & (manifestos['alignment']=='right-wing')])}")
print("--"*10)
print(f"Total manifestos for Spain {len(manifestos[(manifestos['areaServed']=='http://w3id.org/podio/resource/ES')])}")
print(f"Left-wing manifestos for Spain {len(manifestos[(manifestos['areaServed']=='http://w3id.org/podio/resource/ES') & (manifestos['alignment']=='left-wing')])}")
print(f"Centrism manifestos for Spain {len(manifestos[(manifestos['areaServed']=='http://w3id.org/podio/resource/ES') & (manifestos['alignment']=='centrism')])}")
print(f"Right-wing manifestos for Spain {len(manifestos[(manifestos['areaServed']=='http://w3id.org/podio/resource/ES') & (manifestos['alignment']=='right-wing')])}")

In [ ]:
proposals= get_proposals_info()
print(f"Total proposals {len(proposals)}")
print("--"*10)
print(f"Total proposals for France {len(proposals[(proposals['areaServed']=='http://w3id.org/podio/resource/FR')])}")
print(f"Left-wing proposals for France {len(proposals[(proposals['areaServed']=='http://w3id.org/podio/resource/FR') & (proposals['alignment']=='left-wing')])}")
print(f"Centrism proposals for France {len(proposals[(proposals['areaServed']=='http://w3id.org/podio/resource/FR') & (proposals['alignment']=='centrism')])}")
print(f"Right-wing proposals for France {len(proposals[(proposals['areaServed']=='http://w3id.org/podio/resource/FR') & (proposals['alignment']=='right-wing')])}")
print("--"*10)
print(f"Total proposals for Spain {len(proposals[(proposals['areaServed']=='http://w3id.org/podio/resource/ES')])}")
print(f"Left-wing proposals for Spain {len(proposals[(proposals['areaServed']=='http://w3id.org/podio/resource/ES') & (proposals['alignment']=='left-wing')])}")
print(f"Centrism proposals for Spain {len(proposals[(proposals['areaServed']=='http://w3id.org/podio/resource/ES') & (proposals['alignment']=='centrism')])}")
print(f"Right-wing proposals for Spain {len(proposals[(proposals['areaServed']=='http://w3id.org/podio/resource/ES') & (proposals['alignment']=='right-wing')])}")